# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane:** Lane 2 – Refresh / Content Opportunity Scoring.

**The contract in plain words (5 answers):**

1. **What one row means:** one content page (`content_hash_id`) at one decision moment. In the raw table, one row is one page on one day (`report_date` × `client_hash_id` × `content_hash_id`). I roll those daily rows up to one row per page.
2. **Table(s):** `fact_content_daily_performance`, partition `month=2026-03` (a mid-panel month). `dim_clients` is only used to read history coverage.
3. **Time window:** the **decision moment is the end of 15 March 2026**.
   - Features come only from **1–15 March** (the past).
   - The label comes only from **16–31 March** (the future).
   - The final month (June 2026) stays sealed as a test month and is not touched here.
4. **What I rank (label / proxy):** `is_declining` = 1 if a page's average daily impressions in 16–31 March drop below **80%** of its average daily impressions in 1–15 March. The label only counts pages with at least 50 impressions in the first half, so tiny pages don't create noise. The model ranks pages by how likely they are to decline, the same idea as my Week 2 framing.
5. **Deliberately excluded:** anything from 16–31 March as a feature (it is the future). IDs (`client_hash_id`, `content_hash_id`) are used only for joining and splitting, never as features. All GA4 columns are left out for now, because GA4 data is only available for some clients and days and zeros there often mean "not tracked", not "no visits".

In [1]:
import os, getpass
import duckdb
import numpy as np
import pandas as pd

# Token: environment variable, then the local Hugging Face login, then a safe prompt. Never pasted into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from huggingface_hub import get_token
        HF_TOKEN = get_token()
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DECISION_DATE = "2026-03-15"   # features <= this date, label > this date
print("Connected. Working on month", MONTH)

Connected. Working on month 2026-03


C:\Users\Itachi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` (1–15 Mar) | **Feature** (after aggregation) | Observed before the decision moment |
| `gsc_impressions` (16–31 Mar) | **Label** | The label is built from it, so it can never be a feature |
| `report_date` | **Context** | Used to split the past window from the future window |
| `client_hash_id`, `content_hash_id` | **Context** | Joining and the client-grouped split only; pseudonyms carry no real meaning |
| `gsc_data_available` | **Context** (filter) | Keep only rows where it `IS TRUE` |
| All `ga4_*`, `sessions_*`, `ai_*`, `scroll_events` | **Excluded** | Only filled for some clients and days; zero-filled rows would look like "no traffic" |
| Anything dated 16–31 Mar | **Excluded as a feature** | It is future information at the decision moment |
| `month` | **Excluded** | The same value on every row here |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 – Grain: is one raw row really one page on one day?
If the grain holds, no (`report_date`, `client_hash_id`, `content_hash_id`) combination appears more than once.

In [2]:
grain = con.sql(f"""
    SELECT COUNT(*) AS duplicate_keys
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM {FACT}
        GROUP BY ALL
        HAVING c > 1
    )
""").df()
grain

,duplicate_keys
0,0


### Query 2 – Row count and date span of my slice

In [3]:
span = con.sql(f"""
    SELECT COUNT(*)                        AS rows,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT report_date)     AS days,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS pages
    FROM {FACT}
""").df()
span

,rows,first_day,last_day,days,clients,pages
0,9841378,2026-03-01,2026-03-31,31,55,331437


### Query 3 – Availability: how many rows survive `gsc_data_available IS TRUE`?

In [4]:
avail = con.sql(f"""
    SELECT COUNT(*)                                              AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)    AS gsc_available_rows,
           ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_surviving,
           COUNT(DISTINCT client_hash_id) FILTER (WHERE gsc_data_available IS TRUE)    AS clients_surviving
    FROM {FACT}
""").df()
avail

,all_rows,gsc_available_rows,pct_surviving,clients_surviving
0,9841378,3611061,36.7,47


**What the three queries show:** the grain holds (0 duplicate keys). The slice covers all 31 days of March 2026. Only about a third of the raw rows have `gsc_data_available IS TRUE`, so everything below uses only those rows. Without that filter, a zero would mean "not tracked" and would pretend to be "no impressions".

### Five features (max)

Each feature comes only from 1–15 March, on `gsc_data_available IS TRUE` rows. The label comes only from 16–31 March.

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `log_imp_h1`: log of total impressions, 1–15 Mar | it only counts impressions that already happened by 15 March |
| 2 | `ctr_h1`: clicks ÷ impressions, 1–15 Mar | both parts are Search Console numbers already reported by 15 March |
| 3 | `avg_pos_h1`: impression-weighted average position, 1–15 Mar | it is built from `gsc_sum_position` and impressions up to 15 March |
| 4 | `active_days_h1`: days with at least 1 impression, 1–15 Mar | it counts past days only |
| 5 | `momentum_h1`: daily impressions in 8–15 Mar ÷ daily impressions in 1–7 Mar | both weeks are fully before the decision moment |

In [5]:
frame = con.sql(f"""
    WITH d AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_sum_position
        FROM {FACT}
        WHERE gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
        -- past window (features)
        SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '{DECISION_DATE}') AS imp_h1,
        SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '{DECISION_DATE}') AS clk_h1,
        SUM(gsc_sum_position) FILTER (WHERE report_date <= DATE '{DECISION_DATE}') AS sumpos_h1,
        COUNT(*) FILTER (WHERE report_date <= DATE '{DECISION_DATE}' AND gsc_impressions > 0) AS active_days_h1,
        SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-07') AS imp_wk1,
        SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '2026-03-07' AND report_date <= DATE '{DECISION_DATE}') AS imp_wk2,
        -- future window (label only)
        SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '{DECISION_DATE}') AS imp_h2
    FROM d
    GROUP BY 1, 2
    HAVING imp_h1 >= 50
""").df().fillna({"imp_h2": 0, "imp_wk1": 0, "imp_wk2": 0})

frame["log_imp_h1"]  = np.log1p(frame["imp_h1"])
frame["ctr_h1"]      = frame["clk_h1"] / frame["imp_h1"]
frame["avg_pos_h1"]  = frame["sumpos_h1"] / frame["imp_h1"]
frame["momentum_h1"] = (frame["imp_wk2"] / 8) / ((frame["imp_wk1"] / 7) + 1)

# Label: average daily impressions in 16-31 Mar (16 days) below 80% of 1-15 Mar (15 days)
frame["is_declining"] = ((frame["imp_h2"] / 16) < 0.8 * (frame["imp_h1"] / 15)).astype(int)

FEATURES = ["log_imp_h1", "ctr_h1", "avg_pos_h1", "active_days_h1", "momentum_h1"]
print(f"Pages in the feature frame: {len(frame):,}  |  clients: {frame['client_hash_id'].nunique()}")
print(f"Declining base rate: {frame['is_declining'].mean():.3f}")
frame[FEATURES + ["is_declining"]].describe().round(3)

Pages in the feature frame: 92,548  |  clients: 40
Declining base rate: 0.327


,log_imp_h1,ctr_h1,avg_pos_h1,active_days_h1,momentum_h1,is_declining
count,92548.000,92548.000,92548.000,92548.000,92548.000,92548.000
mean,6.135,0.003,13.687,14.049,2.611,0.327
std,1.404,0.005,14.506,2.216,15.799,0.469
min,3.932,0.000,0.000,1.000,0.000,0.000
25%,4.970,0.000,4.459,14.000,0.700,0.000
50%,6.009,0.001,7.773,15.000,0.927,0.000
75%,7.117,0.004,17.657,15.000,1.250,1.000
max,11.993,0.214,119.430,15.000,1697.000,1.000


### Quick honest score

The train/test split is **grouped by client**, so the model is always tested on clients it never saw during training. I report ROC AUC and Precision@50, the metric from my Week 2 framing.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(split.split(frame, frame["is_declining"], groups=frame["client_hash_id"]))
train, test = frame.iloc[train_idx], frame.iloc[test_idx]

def quick_score(cols):
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1)
    model.fit(train[cols], train["is_declining"])
    p = model.predict_proba(test[cols])[:, 1]
    top50 = test.assign(p=p).nlargest(50, "p")
    return roc_auc_score(test["is_declining"], p), top50["is_declining"].mean()

print(f"Test base rate: {test['is_declining'].mean():.3f}  (random Precision@50 would be about this)")
honest_auc, honest_p50 = quick_score(FEATURES)
print(f"HONEST  (5 features):   ROC AUC = {honest_auc:.3f}   Precision@50 = {honest_p50:.2f}")

Test base rate: 0.405  (random Precision@50 would be about this)


HONEST  (5 features):   ROC AUC = 0.754   Precision@50 = 0.96


### The trap: adding one label-derived column on purpose

I add `imp_h2`, the impressions from 16–31 March. That is the exact column the label is computed from. At the decision moment (15 March) this number does not exist yet.

In [7]:
leaky_auc, leaky_p50 = quick_score(FEATURES + ["imp_h2"])
print(f"LEAKY   (+ imp_h2):     ROC AUC = {leaky_auc:.3f}   Precision@50 = {leaky_p50:.2f}")
print(f"HONEST  (5 features):   ROC AUC = {honest_auc:.3f}   Precision@50 = {honest_p50:.2f}")

LEAKY   (+ imp_h2):     ROC AUC = 0.996   Precision@50 = 1.00
HONEST  (5 features):   ROC AUC = 0.754   Precision@50 = 0.96


The score jumps toward perfect as soon as `imp_h2` is added. The model isn't smarter: it is reading the answer, because the label is literally `imp_h2` compared against `imp_h1`. In real use, 16–31 March hasn't happened yet on 15 March, so that score could never be reproduced.

**I delete the leaky column and keep the honest number.**

In [8]:
frame = frame.drop(columns=["imp_h2"])   # removed for good; the label stays, the leak goes
assert "imp_h2" not in frame.columns and "imp_h2" not in FEATURES
print(f"Kept honest score -> ROC AUC = {honest_auc:.3f}, Precision@50 = {honest_p50:.2f}")

Kept honest score -> ROC AUC = 0.754, Precision@50 = 0.96


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation of my slice:** it is **a single month, split into two short halves**. A two-week-vs-two-week impression change is noisy. Things like a holiday, a Google update, or one viral day can flip the label, and March may not behave like other months. So the honest score above is directional evidence for this one month only, not a claim that the model works all year.

Other limits I can see from the queries:
- Only about a third of raw rows have `gsc_data_available IS TRUE`, so pages from clients without Search Console data are simply not in this frame. The results say nothing about them.
- The panel is unbalanced: clients started tracking at different dates (checked below), so a longer window would cover different clients unevenly.
- The honest Precision@50 of 0.96 comes from one client-grouped split. The top 50 are most likely pages whose impressions were already falling in 8–15 March (low `momentum_h1`), so they simply kept falling. That is real, usable signal, but it is the easy end of the list. ROC AUC (0.754) is the fairer summary of the whole ranking, and the number needs repeated splits before I trust it.
- Search Console only records pages that got impressions. A page with zero impressions is invisible, not measured as zero.

In [9]:
clients = con.sql(f"""
    SELECT access_profile,
           COUNT(*)            AS clients,
           MIN(gsc_data_start) AS earliest_gsc_start,
           MAX(gsc_data_start) AS latest_gsc_start
    FROM read_parquet('{REL}/dim_clients.parquet')
    GROUP BY 1
    ORDER BY clients DESC
""").df()
clients

,access_profile,clients,earliest_gsc_start,latest_gsc_start
0,gsc_and_ga4,53,2025-01-27,2026-06-02
1,no_search_or_analytics_access,26,2025-11-05,2025-11-05
2,gsc_only,14,2025-06-07,2026-04-07
3,source_only_missing_client_dimension,10,2025-11-05,2025-11-07
4,ga4_only,1,NaT,NaT


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.